# RQ2: Regression — Predicting Campaign Revenue

**Research Question:** Which campaign and customer features best predict Revenue_Generated, and how accurately can machine learning regression models forecast campaign revenue?

**Task:** Regression  
**Target:** `Revenue_Generated` (continuous USD)  
**Models:** Ridge Regression, Random Forest Regressor, XGBoost Regressor  
**Dataset:** Marketing and Product Performance Dataset (Kaggle)

In [29]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
sns.set_palette(PALETTE)
RANDOM_STATE = 42
OUTPUT_DIR = '/kaggle/working/'

def save_figure(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved figure: {path}')

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'Saved table:  {path}')

def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def adj_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

print('Imports OK')

Imports OK


## 1. Data Loading

In [30]:
input_dir = '/kaggle/input'
data_files = [
    os.path.join(root, f)
    for root, dirs, files in os.walk(input_dir)
    for f in files if f.endswith('.xlsx') or f.endswith('.xls') or f.endswith('.csv')
]
print('Found files:', data_files)
FILE_PATH = data_files[0]

df = pd.read_csv(FILE_PATH) if FILE_PATH.endswith('.csv') else pd.read_excel(FILE_PATH)
print(f'Shape: {df.shape}')
df.head()

Found files: ['/kaggle/input/datasets/vanishjr/marketing-product-performance/marketing_and_product_performance.csv']
Shape: (10000, 17)


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative
2,CMP_6SBOWN,PROD_4V8A08,22367.45,3546,265,32585.62,1.41,CUST_BR2GST,Basic,9,FLASH_J4PEON,51,160,BNDL_0HY0EF,462.87,4,Affordable
3,CMP_Q31QCU,PROD_A1Q6ZB,29957.54,2573,781,95740.12,3.32,CUST_6TBY6K,Premium,32,FLASH_1TOVXT,36,159,BNDL_AI09BC,334.16,1,Durable
4,CMP_AY0UTJ,PROD_F57N66,36277.19,818,79,81990.43,3.53,CUST_XASI45,Standard,29,FLASH_AOBHXL,20,52,BNDL_R03ITT,371.67,2,Affordable


## 2. Feature Engineering

In [31]:
df['Has_Flash_Sale'] = df['Flash_Sale_ID'].notna().astype(int)
df['Has_Bundle']     = df['Bundle_ID'].notna().astype(int)
df['Conversion_Rate']    = np.where(df['Clicks'] > 0, df['Conversions'] / df['Clicks'], 0)
df['Cost_Per_Conversion'] = np.where(df['Conversions'] > 0, df['Budget'] / df['Conversions'], 0)
df['Spend_Ratio'] = np.where(df['Budget'] > 0, df['Conversions'] / df['Budget'], 0)  # conversions per dollar

# Drop leakage: ROI is derived from Revenue_Generated
NUMERIC_FEATURES = [
    'Budget', 'Clicks', 'Conversions', 'Discount_Level', 'Units_Sold',
    'Bundle_Price', 'Subscription_Length', 'Has_Flash_Sale', 'Has_Bundle',
    'Conversion_Rate', 'Cost_Per_Conversion', 'Spend_Ratio',
    'Customer_Satisfaction_Post_Refund'
]
CAT_FEATURES = ['Subscription_Tier']

X = df[NUMERIC_FEATURES + CAT_FEATURES].copy()
y = df['Revenue_Generated'].copy()

print(f'Target stats — min: {y.min():.2f}  max: {y.max():.2f}  mean: {y.mean():.2f}  skew: {y.skew():.2f}')

Target stats — min: 1002.08  max: 99999.47  mean: 50038.63  skew: 0.03


In [32]:
# Apply log1p transform if skewness > 1
USE_LOG = y.skew() > 1.0
print(f'Apply log1p transform: {USE_LOG}')
y_model = np.log1p(y) if USE_LOG else y

Apply log1p transform: False


## 3. EDA

In [33]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y, bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
y.plot.kde(ax=ax, secondary_y=False, color=PALETTE[1], linewidth=2)
ax.set_title('Revenue_Generated Distribution (RQ2)', fontsize=13, fontweight='bold')
ax.set_xlabel('Revenue Generated (USD)')
ax.set_ylabel('Frequency')
plt.tight_layout()
save_figure(fig, 'rq2_revenue_distribution.pdf')
plt.show()

Saved figure: /kaggle/working/rq2_revenue_distribution.pdf


In [34]:
# Feature–Revenue Pearson correlation bar chart
num_df = df[NUMERIC_FEATURES + ['Revenue_Generated']].dropna()
corrs = {col: pearsonr(num_df[col], num_df['Revenue_Generated'])[0] for col in NUMERIC_FEATURES}
corr_series = pd.Series(corrs).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [PALETTE[0] if v >= 0 else PALETTE[1] for v in corr_series.values]
corr_series.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation with Revenue_Generated (RQ2)', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Pearson r')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
save_figure(fig, 'rq2_feature_revenue_correlation.pdf')
plt.show()

Saved figure: /kaggle/working/rq2_feature_revenue_correlation.pdf


## 4. Preprocessing Pipeline

In [35]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', cat_transformer, CAT_FEATURES)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_model, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Train: (8000, 14) | Test: (2000, 14)


## 5. Model Training & Cross-Validation

In [36]:
models = {
    'Ridge':         RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]),
    'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':       XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  random_state=RANDOM_STATE, verbosity=0)
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
fitted_models = {}

for name, reg in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', reg)])
    t0 = time.time()
    cv_r2 = cross_validate(pipe, X_train, y_train, cv=cv, scoring='r2')
    pipe.fit(X_train, y_train)
    train_time = round(time.time() - t0, 2)
    fitted_models[name] = pipe

    y_pred_log = pipe.predict(X_test)
    y_pred     = np.expm1(y_pred_log) if USE_LOG else y_pred_log
    y_true     = np.expm1(y_test)     if USE_LOG else y_test

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    ar2  = adj_r2(r2, len(y_true), X_test.shape[1])
    mp   = mape(y_true.values, y_pred)

    results.append({
        'Model': name, 'RMSE': round(rmse, 4), 'MAE': round(mae, 4),
        'R2': round(r2, 4), 'Adj_R2': round(ar2, 4), 'MAPE_%': round(mp, 2),
        'CV_R2_Mean': round(cv_r2['test_score'].mean(), 4),
        'CV_R2_Std':  round(cv_r2['test_score'].std(), 4),
        'Train_Time_s': train_time
    })
    print(f'{name}: R²={results[-1]["R2"]}  RMSE={results[-1]["RMSE"]:.2f}  MAPE={results[-1]["MAPE_%"]}%')

results_df = pd.DataFrame(results)
save_table(results_df, 'rq2_model_comparison.csv')
results_df

Ridge: R²=-0.0025  RMSE=28591.24  MAPE=166.03%
Random Forest: R²=-0.0161  RMSE=28785.32  MAPE=166.34%
XGBoost: R²=-0.0457  RMSE=29200.66  MAPE=166.62%
Saved table:  /kaggle/working/rq2_model_comparison.csv


,Model,RMSE,MAE,R2,Adj_R2,MAPE_%,CV_R2_Mean,CV_R2_Std,Train_Time_s
0,Ridge,28591.2450,24747.7549,-0.0025,-0.0096,166.03,-0.0025,0.0030,0.32
1,Random Forest,28785.3205,24892.3133,-0.0161,-0.0233,166.34,-0.0126,0.0041,56.92
2,XGBoost,29200.6639,25125.9391,-0.0457,-0.0531,166.62,-0.0598,0.0102,4.41


## 6. Publication-Ready Figures

In [37]:
best_name = results_df.loc[results_df['R2'].idxmax(), 'Model']
best_pipe = fitted_models[best_name]
print(f'Best model: {best_name}')

y_pred_log = best_pipe.predict(X_test)
y_pred = np.expm1(y_pred_log) if USE_LOG else y_pred_log
y_true = np.expm1(y_test)     if USE_LOG else y_test
residuals = y_true.values - y_pred

best_r2 = r2_score(y_true, y_pred)

Best model: Ridge


In [38]:
# ── Actual vs Predicted ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_true, y_pred, alpha=0.4, s=20, color=PALETTE[0], edgecolors='none')
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual Revenue (USD)', fontsize=12)
ax.set_ylabel('Predicted Revenue (USD)', fontsize=12)
ax.set_title(f'Actual vs Predicted Revenue — {best_name} (RQ2)', fontsize=13, fontweight='bold')
ax.text(0.05, 0.92, f'R² = {best_r2:.4f}', transform=ax.transAxes,
        fontsize=12, bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
ax.legend(fontsize=10)
plt.tight_layout()
save_figure(fig, 'rq2_actual_vs_predicted.pdf')
plt.show()

Saved figure: /kaggle/working/rq2_actual_vs_predicted.pdf


In [39]:
# ── Residual Plot ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(y_pred, residuals, alpha=0.4, s=20, color=PALETTE[0], edgecolors='none')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Fitted Values (Predicted Revenue)', fontsize=12)
ax.set_ylabel('Residuals', fontsize=12)
ax.set_title(f'Residual Plot — {best_name} (RQ2)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq2_residual_plot.pdf')
plt.show()

Saved figure: /kaggle/working/rq2_residual_plot.pdf


## 7. Conclusions

In [40]:
best_row = results_df.loc[results_df['R2'].idxmax()]
print('=' * 60)
print('RQ2 CONCLUSION')
print('=' * 60)
print(f'Best model: {best_row["Model"]}')
print(f'  R²    : {best_row["R2"]}')
print(f'  RMSE  : {best_row["RMSE"]}')
print(f'  MAPE  : {best_row["MAPE_%"]}%')
print()
print('Outputs saved:')
for f in ['rq2_revenue_distribution.pdf','rq2_feature_revenue_correlation.pdf',
          'rq2_actual_vs_predicted.pdf','rq2_residual_plot.pdf','rq2_model_comparison.csv']:
    print(f'  {f}')

RQ2 CONCLUSION
Best model: Ridge
  R²    : -0.0025
  RMSE  : 28591.245
  MAPE  : 166.03%

Outputs saved:
  rq2_revenue_distribution.pdf
  rq2_feature_revenue_correlation.pdf
  rq2_actual_vs_predicted.pdf
  rq2_residual_plot.pdf
  rq2_model_comparison.csv
